# v1: Keytel + 반올림 + 역공학 분석

대회 종료 후 상위팀 접근법을 바탕으로 한 개선 실험입니다.

## 실험 구성
1. **Keytel 직접 예측** — ML 없이 생리학 공식 하나로 기준선 설정
2. **반올림(Rounding)** — 정답이 정수 형태임을 활용 (단독으로 ~0.1 감소)
3. **역공학 분석** — 선형 회귀로 공식 계수를 데이터에서 직접 복원
4. **계수 최적화** — scipy로 더 정확한 Keytel 계수 탐색
5. **잔차 ML** — Keytel 오차를 CatBoost로 보정
6. **CatBoost + Keytel Feature + 반올림** — 상위팀 핵심 전략
7. **다중 시드 앙상블** — 복수 시드로 학습 후 평균, 분산 감소
8. **최소 변수 실험 (4개)** — 핵심 변수만으로도 충분한지 검증
9. **SelectKBest(k=5) + Linear Regression** — Keytel 기반 Linear Regression 접근
10. **전략 비교 및 최종 제출**

In [ ]:
import random, os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest, f_regression
from catboost import CatBoostRegressor


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## 1. 데이터 로드

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
submission = pd.read_csv('../data/sample_submission.csv')

train_x = train.drop(['ID', 'Calories_Burned'], axis=1)
train_y = train['Calories_Burned']
test_x  = test.drop(['ID'], axis=1)

print(f'Train: {train_x.shape}, Test: {test_x.shape}')
print(f'\n정답 통계:')
print(train_y.describe())
print(f'\n정답이 정수값인지 확인: {(train_y % 1 == 0).all()}')

## 2. Keytel 공식 직접 예측

Keytel 공식은 심박수·체중·나이·운동시간으로 칼로리를 추정하는 생리학 공식입니다.

| 성별 | 공식 |
|---|---|
| 남성 | `(-55.0969 + 0.6309×HR + 0.1988×W + 0.2017×A) × T / 4.184` |
| 여성 | `(-20.4022 + 0.4472×HR − 0.1263×W + 0.074×A) × T / 4.184` |

W = 체중(kg), A = 나이(세), T = 운동 시간(분)

> **포인트**: 데이터의 체중 단위가 lb이므로 → kg 변환 필수 (`× 0.453592`)

In [ ]:
def compute_keytel(df):
    df = df.copy()
    df['Weight_kg']  = df['Weight(lb)'] * 0.453592
    df['Gender_num'] = df['Gender'].map({'M': 0, 'F': 1})

    male_cal = (
        -55.0969
        + 0.6309 * df['BPM']
        + 0.1988 * df['Weight_kg']
        + 0.2017 * df['Age']
    ) * df['Exercise_Duration'] / 4.184

    female_cal = (
        -20.4022
        + 0.4472 * df['BPM']
        - 0.1263 * df['Weight_kg']
        + 0.074  * df['Age']
    ) * df['Exercise_Duration'] / 4.184

    df['Calories_Keytel'] = np.where(df['Gender_num'] == 0, male_cal, female_cal)
    df['Calories_Keytel'] = np.maximum(df['Calories_Keytel'], 0)
    return df


train_feat   = compute_keytel(train_x)
keytel_pred   = train_feat['Calories_Keytel']
keytel_rounded = np.round(keytel_pred)

rmse_keytel  = root_mean_squared_error(train_y, keytel_pred)
rmse_rounded = root_mean_squared_error(train_y, keytel_rounded)

print(f'Keytel 직접 예측 RMSE : {rmse_keytel:.4f}')
print(f'Keytel + 반올림 RMSE  : {rmse_rounded:.4f}')
print(f'반올림만으로 감소한 RMSE: {rmse_keytel - rmse_rounded:.4f}')

## 3. 역공학 분석 (Reverse Engineering)

### 핵심 가설
> 실제 정답 = `round(Keytel(HR, Weight_kg, Age, Duration))`

이 가설이 맞다면 잔차가 매우 작고, 반올림 정확 일치율이 높아야 합니다.

### 3a. 잔차 분포 분석

In [ ]:
residuals = train_y - keytel_pred

exact_match = (keytel_rounded == train_y).mean()
within_half = (residuals.abs() < 0.5).mean()
within_one  = (residuals.abs() < 1.0).mean()

print('잔차(실제값 - Keytel 예측값) 통계:')
print(residuals.describe().round(4))
print(f'\n반올림 정확 일치율       : {exact_match:.2%}')
print(f'|잔차| < 0.5 비율        : {within_half:.2%}')
print(f'|잔차| < 1.0 비율        : {within_one:.2%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(residuals, bins=100, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Keytel 잔차 분포')
axes[0].set_xlabel('실제값 - Keytel 예측값')

axes[1].scatter(keytel_pred, residuals, alpha=0.2, s=5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('예측값 크기별 잔차')
axes[1].set_xlabel('Keytel 예측값 (칼로리)')
axes[1].set_ylabel('잔차')

plt.tight_layout()
plt.show()

In [ ]:
male_mask   = train_feat['Gender_num'] == 0
female_mask = train_feat['Gender_num'] == 1

print(f"{'성별':<8} {'RMSE':<12} {'반올림 RMSE':<16} {'정확 일치율':<15} {'샘플 수'}")
print('-' * 60)
for label, mask in [('남성', male_mask), ('여성', female_mask)]:
    rmse_g   = root_mean_squared_error(train_y[mask], keytel_pred[mask])
    rmse_g_r = root_mean_squared_error(train_y[mask], keytel_rounded[mask])
    match    = (keytel_rounded[mask] == train_y[mask]).mean()
    print(f'{label:<8} {rmse_g:<12.4f} {rmse_g_r:<16.4f} {match:<15.2%} {mask.sum()}')

### 3b. 선형 회귀 역공학 — 데이터에서 공식 계수 직접 복원

Keytel 공식을 모른다는 가정 하에, 데이터에서 선형 회귀로 계수를 역추적합니다.  
복원된 계수 ≈ Keytel 표준 계수이면 → **데이터가 Keytel로 생성됐다는 증거**

In [ ]:
keytel_standard = {
    'M': {'intercept': -55.0969, 'BPM': 0.6309, 'Weight_kg': 0.1988, 'Age': 0.2017},
    'F': {'intercept': -20.4022, 'BPM': 0.4472, 'Weight_kg': -0.1263, 'Age': 0.074},
}

print('=' * 65)
print('선형 회귀 역공학 계수 vs Keytel 표준 계수')
print('=' * 65)

for gender_label, mask, key in [('남성(M)', male_mask, 'M'), ('여성(F)', female_mask, 'F')]:
    X_lr = train_feat[['BPM', 'Weight_kg', 'Age', 'Exercise_Duration']][mask].copy()
    # Keytel 구조: (a + b*BPM + c*W + d*A) * T / 4.184
    X_lr_div = X_lr.copy()
    for col in ['BPM', 'Weight_kg', 'Age']:
        X_lr_div[col] = X_lr[col] * X_lr['Exercise_Duration'] / 4.184
    X_lr_div['bias'] = X_lr['Exercise_Duration'] / 4.184

    lr = LinearRegression(fit_intercept=False)
    lr.fit(X_lr_div[['bias', 'BPM', 'Weight_kg', 'Age', 'Exercise_Duration']], train_y[mask])

    pred = np.maximum(lr.predict(X_lr_div[['bias', 'BPM', 'Weight_kg', 'Age', 'Exercise_Duration']]), 0)
    rmse_lr_r = root_mean_squared_error(train_y[mask], np.round(pred))

    std = keytel_standard[key]
    print(f'\n[{gender_label}]  반올림 RMSE: {rmse_lr_r:.4f}')
    print(f"  {'계수':<15} {'역공학값':>12}  {'Keytel 표준값':>14}")
    print(f"  {'상수(intercept)':<15} {lr.coef_[0]:>12.4f}  {std['intercept']:>14.4f}")
    print(f"  {'BPM':<15} {lr.coef_[1]:>12.4f}  {std['BPM']:>14.4f}")
    print(f"  {'Weight_kg':<15} {lr.coef_[2]:>12.4f}  {std['Weight_kg']:>14.4f}")
    print(f"  {'Age':<15} {lr.coef_[3]:>12.4f}  {std['Age']:>14.4f}")
    print(f"  {'Duration 잔여':<15} {lr.coef_[4]:>12.6f}  (이상적으로 ≈ 0)")

### 3c. 계수 최적화 — scipy로 반올림 RMSE를 직접 최소화

In [ ]:
def keytel_custom(params, df):
    a_m, b_m, c_m, d_m, a_f, b_f, c_f, d_f, divisor = params
    cal = np.where(
        df['Gender_num'] == 0,
        (a_m + b_m * df['BPM'] + c_m * df['Weight_kg'] + d_m * df['Age']) * df['Exercise_Duration'] / divisor,
        (a_f + b_f * df['BPM'] + c_f * df['Weight_kg'] + d_f * df['Age']) * df['Exercise_Duration'] / divisor
    )
    return np.maximum(cal, 0)

def objective_round(params):
    pred = keytel_custom(params, train_feat)
    return root_mean_squared_error(train_y, np.round(pred))

initial = [-55.0969, 0.6309, 0.1988, 0.2017, -20.4022, 0.4472, -0.1263, 0.074, 4.184]

print('계수 최적화 중... (수십 초 소요)')
result = minimize(objective_round, initial, method='Nelder-Mead',
                  options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-8})

opt_pred   = keytel_custom(result.x, train_feat)
rmse_opt_r = root_mean_squared_error(train_y, np.round(opt_pred))

print(f'\n표준 Keytel + 반올림 RMSE   : {rmse_rounded:.4f}')
print(f'최적화 Keytel + 반올림 RMSE : {rmse_opt_r:.4f}')
names = ['남성_상수', '남성_HR', '남성_Weight', '남성_Age', '여성_상수', '여성_HR', '여성_Weight', '여성_Age', '제수']
print(f"\n{'계수':<15} {'최적화값':>14}  {'표준값':>14}  {'차이':>10}")
print('-' * 58)
for name, opt_v, std_v in zip(names, result.x, initial):
    print(f'{name:<15} {opt_v:>14.6f}  {std_v:>14.6f}  {opt_v - std_v:>10.6f}')

## 4. 잔차 ML 모델링

Keytel이 맞추지 못한 오차를 CatBoost로 보정합니다.

```
최종 예측 = round(Keytel(X) + CatBoost_보정(X))
```

In [ ]:
def build_features(df):
    df = compute_keytel(df)
    df['Height_cm']     = df['Height(Feet)'] * 30.48 + df['Height(Remainder_Inches)'] * 2.54
    df['BMI']           = df['Weight_kg'] / ((df['Height_cm'] / 100) ** 2)
    df['Max_HR']        = 208 - 0.7 * df['Age']
    df['HR_Ratio']      = df['BPM'] / df['Max_HR']
    df['Temp_C']        = (df['Body_Temperature(F)'] - 32) * 5 / 9
    df['Weight_Status'] = df['Weight_Status'].map(
        {'Normal Weight': 0, 'Overweight': 1, 'Obese': 2})
    drop_cols = ['Height(Feet)', 'Height(Remainder_Inches)', 'Weight(lb)',
                 'Body_Temperature(F)', 'Gender']
    df = df.drop(columns=drop_cols, errors='ignore')
    return df

train_built = build_features(train_x)
test_built  = build_features(test_x)

all_feature_cols = [c for c in train_built.columns if c not in ['Gender_num']]
print(f'전체 Feature ({len(all_feature_cols)}개): {all_feature_cols}')

In [ ]:
y_residual = train_y - train_built['Calories_Keytel']
print(f'잔차 범위: {y_residual.min():.2f} ~ {y_residual.max():.2f}')

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_residual = np.zeros(len(train_y))

# Keytel 자체는 잔차 모델에서 제외 (target 누수 방지)
resid_cols = [c for c in all_feature_cols if c != 'Calories_Keytel']
X_resid    = train_built[resid_cols]

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_resid)):
    X_tr, X_val = X_resid.iloc[tr_idx], X_resid.iloc[val_idx]
    y_tr        = y_residual.iloc[tr_idx]
    y_val_r     = y_residual.iloc[val_idx]

    model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               random_state=42, verbose=0)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val_r), early_stopping_rounds=50)
    oof_residual[val_idx] = model.predict(X_val)

final_res_pred  = train_built['Calories_Keytel'] + oof_residual
final_res_round = np.round(np.maximum(final_res_pred, 0))

rmse_res       = root_mean_squared_error(train_y, final_res_pred)
rmse_res_round = root_mean_squared_error(train_y, final_res_round)

print(f'Keytel + 잔차 ML RMSE        : {rmse_res:.4f}')
print(f'Keytel + 잔차 ML + 반올림   : {rmse_res_round:.4f}')

## 5. CatBoost + Keytel Feature + 반올림

Keytel 예측값을 feature로 포함한 CatBoost + 최종 반올림

In [ ]:
X_all = train_built[all_feature_cols]
oof_cat = np.zeros(len(train_y))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_all)):
    X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
    y_tr, y_val = train_y.iloc[tr_idx], train_y.iloc[val_idx]

    cat = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6,
                             random_state=42, verbose=0)
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
    oof_cat[val_idx] = cat.predict(X_val)
    print(f'Fold {fold+1} RMSE: {root_mean_squared_error(y_val, cat.predict(X_val)):.4f}')

rmse_cat       = root_mean_squared_error(train_y, oof_cat)
rmse_cat_round = root_mean_squared_error(train_y, np.round(np.maximum(oof_cat, 0)))

print(f'\nCatBoost + Keytel Feature RMSE     : {rmse_cat:.4f}')
print(f'CatBoost + Keytel Feature + 반올림 : {rmse_cat_round:.4f}')

## 6. 다중 시드 앙상블 (Multi-seed)

동일한 KFold 분할(고정)에서 **모델 시드만 다르게** 여러 번 학습하고 평균냅니다.

- 모델의 초기화 랜덤성에 의한 분산을 줄여 안정적인 예측 확보
- 단독으로는 작은 효과지만, 반올림과 결합 시 미세한 경계값 처리에 유리

In [ ]:
seeds = [0, 42, 100, 200, 314]
all_oof_preds = np.zeros((len(seeds), len(train_y)))

kf_fixed = KFold(n_splits=5, shuffle=True, random_state=42)  # 분할 고정

for i, seed in enumerate(seeds):
    oof_s = np.zeros(len(train_y))

    for fold, (tr_idx, val_idx) in enumerate(kf_fixed.split(X_all)):
        X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
        y_tr, y_val = train_y.iloc[tr_idx], train_y.iloc[val_idx]

        cat = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6,
                                 random_state=seed, verbose=0)  # seed만 변경
        cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
        oof_s[val_idx] = cat.predict(X_val)

    all_oof_preds[i] = oof_s
    r = root_mean_squared_error(train_y, np.round(np.maximum(oof_s, 0)))
    print(f'Seed {seed:>4}: 반올림 RMSE = {r:.4f}')

# 시드 평균 예측
multi_seed_pred  = np.mean(all_oof_preds, axis=0)
rmse_ms          = root_mean_squared_error(train_y, multi_seed_pred)
rmse_ms_round    = root_mean_squared_error(train_y, np.round(np.maximum(multi_seed_pred, 0)))

print(f'\n다중 시드 앙상블 RMSE       : {rmse_ms:.4f}')
print(f'다중 시드 앙상블 + 반올림   : {rmse_ms_round:.4f}')

## 7. 최소 변수 실험 (4개 변수)

다른 팀 발표에서 **변수 4개만으로 0.07** 달성 사례가 있었습니다.

Keytel 공식과 동일한 핵심 변수 4개 + 성별로만 구성:
- `BPM`, `Weight_kg`, `Age`, `Exercise_Duration`, `Gender_num`

복잡한 feature engineering 없이도 충분한지 확인합니다.

In [ ]:
# 4개 핵심 변수만 (Keytel 공식 구성 변수)
min_cols   = ['BPM', 'Weight_kg', 'Age', 'Exercise_Duration', 'Gender_num']
# + Keytel 추가 버전 (5번째 변수)
min_cols_k = min_cols + ['Calories_Keytel']

results_min = {}

for label, cols in [('4변수', min_cols), ('4변수 + Keytel', min_cols_k)]:
    X_min     = train_built[cols]
    oof_min   = np.zeros(len(train_y))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_min)):
        X_tr, X_val = X_min.iloc[tr_idx], X_min.iloc[val_idx]
        y_tr, y_val = train_y.iloc[tr_idx], train_y.iloc[val_idx]

        cat = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6,
                                 random_state=42, verbose=0)
        cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
        oof_min[val_idx] = cat.predict(X_val)

    r_raw   = root_mean_squared_error(train_y, oof_min)
    r_round = root_mean_squared_error(train_y, np.round(np.maximum(oof_min, 0)))
    results_min[label] = (r_raw, r_round)
    print(f'{label:<20} RMSE: {r_raw:.4f}  |  반올림 RMSE: {r_round:.4f}')

print('\n→ 전체 변수 대비 성능 차이가 작다면, 변수를 줄이는 것이 더 효율적일 수 있습니다.')

## 8. SelectKBest(k=5) + Linear Regression

대회 상위팀 접근 방식 중 하나: Keytel 기반 feature에서 상위 k개만 골라 선형 회귀로 학습합니다.

```
Keytel 기반 feature → SelectKBest(k=5) → LinearRegression → 반올림
```

- 전체 feature에서 F-score 기준 상위 5개만 선택
- LinearRegression으로 학습 → 반올림

**어떤 5개가 선택되는지**가 핵심 — Keytel이 압도적 1위라면 데이터 생성 공식이 Keytel임을 재확인

In [ ]:
X_for_lr = train_built[all_feature_cols]

# 전체 데이터 기준으로 선택된 변수 미리 확인
global_sel = SelectKBest(f_regression, k=5)
global_sel.fit(X_for_lr, train_y)
selected_features = [f for f, m in zip(all_feature_cols, global_sel.get_support()) if m]
scores = global_sel.scores_

score_df = pd.DataFrame({'Feature': all_feature_cols, 'F-score': scores})
score_df = score_df.sort_values('F-score', ascending=False)
print('F-score 기준 전체 순위:')
print(score_df.to_string(index=False))
print(f'\n→ SelectKBest(k=5) 선택 변수: {selected_features}')

In [ ]:
# KFold 교차 검증으로 평가
oof_lr = np.zeros(len(train_y))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_for_lr)):
    X_tr, X_val = X_for_lr.iloc[tr_idx], X_for_lr.iloc[val_idx]
    y_tr, y_val = train_y.iloc[tr_idx], train_y.iloc[val_idx]

    # fold별 feature selection (데이터 누수 방지)
    sel = SelectKBest(f_regression, k=5)
    X_tr_sel  = sel.fit_transform(X_tr, y_tr)
    X_val_sel = sel.transform(X_val)

    lr = LinearRegression()
    lr.fit(X_tr_sel, y_tr)
    oof_lr[val_idx] = lr.predict(X_val_sel)

oof_lr = np.maximum(oof_lr, 0)
rmse_lr       = root_mean_squared_error(train_y, oof_lr)
rmse_lr_round = root_mean_squared_error(train_y, np.round(oof_lr))

print(f'SelectKBest(k=5) + LinReg RMSE         : {rmse_lr:.4f}')
print(f'SelectKBest(k=5) + LinReg + 반올림     : {rmse_lr_round:.4f}')

In [ ]:
# k값 변화에 따른 성능 변화 확인
print('k값별 반올림 RMSE:')
for k in [1, 2, 3, 4, 5, 7, 10]:
    oof_k = np.zeros(len(train_y))
    for tr_idx, val_idx in kf.split(X_for_lr):
        X_tr, X_val = X_for_lr.iloc[tr_idx], X_for_lr.iloc[val_idx]
        y_tr = train_y.iloc[tr_idx]
        sel = SelectKBest(f_regression, k=k)
        lr  = LinearRegression()
        lr.fit(sel.fit_transform(X_tr, y_tr), y_tr)
        oof_k[val_idx] = lr.predict(sel.transform(X_val))
    r = root_mean_squared_error(train_y, np.round(np.maximum(oof_k, 0)))
    print(f'  k={k:>2}: {r:.4f}')

## 9. 전략 비교 및 최종 제출

In [ ]:
results = [
    ('Original (Stacking + SHAP보정)',          0.48),
    ('Keytel 직접 예측',                          rmse_keytel),
    ('Keytel + 반올림',                          rmse_rounded),
    ('최적화 Keytel + 반올림',                    rmse_opt_r),
    ('SelectKBest(k=5) + LinReg + 반올림',       rmse_lr_round),
    ('4변수 CatBoost + 반올림',                   results_min['4변수'][1]),
    ('4변수 + Keytel CatBoost + 반올림',          results_min['4변수 + Keytel'][1]),
    ('Keytel + 잔차 ML + 반올림',                 rmse_res_round),
    ('CatBoost + Keytel Feature + 반올림',        rmse_cat_round),
    ('다중 시드 앙상블 + 반올림',                   rmse_ms_round),
]

results_df = pd.DataFrame(results, columns=['전략', 'CV RMSE'])
results_df = results_df.sort_values('CV RMSE').reset_index(drop=True)

print('=' * 58)
print(f"{'전략':<38} {'CV RMSE':>10}")
print('-' * 58)
for _, row in results_df.iterrows():
    print(f"{row['전략']:<38} {row['CV RMSE']:>10.4f}")
print('=' * 58)

In [ ]:
# [전략 1] Keytel + 반올림 (역공학 기반, 가장 단순)
test_feat = compute_keytel(test_x)
pred_s1   = np.maximum(np.round(test_feat['Calories_Keytel']), 0)

submission['Calories_Burned'] = pred_s1
submission.to_csv('submit_v1_keytel_rounded.csv', index=False)
print(f'[전략 1] submit_v1_keytel_rounded.csv | 평균: {pred_s1.mean():.2f}')

In [ ]:
# [전략 2] SelectKBest(k=5) + LinearRegression + 반올림
X_test_lr  = test_built[all_feature_cols]

# 전체 train으로 재학습
sel_final  = SelectKBest(f_regression, k=5)
X_train_sel = sel_final.fit_transform(X_for_lr, train_y)
X_test_sel  = sel_final.transform(X_test_lr)

lr_final = LinearRegression()
lr_final.fit(X_train_sel, train_y)

pred_s2 = np.maximum(np.round(lr_final.predict(X_test_sel)), 0)
submission['Calories_Burned'] = pred_s2
submission.to_csv('submit_v2_selectkbest_lr.csv', index=False)
print(f'[전략 2] submit_v2_selectkbest_lr.csv | 평균: {pred_s2.mean():.2f}')

In [ ]:
# [전략 3] 다중 시드 앙상블 + 반올림 (전체 데이터로 재학습)
test_all     = test_built[all_feature_cols]
all_test_preds = np.zeros((len(seeds), len(test_all)))

for i, seed in enumerate(seeds):
    cat_ms = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6,
                                random_state=seed, verbose=0)
    cat_ms.fit(X_all, train_y)
    all_test_preds[i] = cat_ms.predict(test_all)
    print(f'Seed {seed} 학습 완료')

pred_s3 = np.maximum(np.round(np.mean(all_test_preds, axis=0)), 0)
submission['Calories_Burned'] = pred_s3
submission.to_csv('submit_v3_multiseed.csv', index=False)
print(f'[전략 3] submit_v3_multiseed.csv | 평균: {pred_s3.mean():.2f}')